# Kontrol v6 pada batas false alarm development yang sama

Notebook evaluasi pendamping; **tidak melatih classifier**. Model v6 dan baseline v3 tetap beku. Lihat [protokol v6](ai-agent-docs/data-improvement-v6-plan.md).

Kontrol ini ditambahkan setelah evaluasi utama dimulai, sebelum hasil test v6 dipakai untuk mendesain kontrol. Ia bukan bagian preregistrasi awal. Threshold referensi v3 dipilih hanya dari threshold partition v6, pada FPR empiris 1%/0,1% per sumber, lalu dibekukan sebelum scoring kontrol. Tidak memakai test untuk memilih threshold, model, atau promosi produksi.

Laporkan ketiga referensi: baseline asli, baseline dengan threshold development yang disamakan, serta v6. Feed PhishTank hanya positif: FPR tidak dapat dihitung dari feed itu; false alarm dilaporkan terpisah pada holdout vendor berlabel historis.

In [1]:
import hashlib
import json
import math
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import joblib
import nbformat
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.sparse import csr_matrix, hstack
from scipy.special import logit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import StandardScaler, normalize
from threadpoolctl import threadpool_limits

NOTEBOOK_NAME = "guardian_lab_url_qr_v6_control.ipynb"
locations = [Path.cwd() / NOTEBOOK_NAME, Path.cwd().parent / NOTEBOOK_NAME]
NOTEBOOK_PATH = next(p.resolve() for p in locations if p.is_file())
ROOT = NOTEBOOK_PATH.parent
RUN = ROOT / "ml/runs/notebook-tfidf-logistic-v6"
REPORTS, DATA = RUN / "reports", RUN / "data"
V3 = ROOT / "ml/runs/notebook-tfidf-logistic-v3"
SEED = 20261101


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )


document = nbformat.read(NOTEBOOK_PATH, as_version=4)
control_code_sha256 = hashlib.sha256(
    "\n\n".join(c.source for c in document.cells if c.cell_type == "code").encode()
).hexdigest()

In [2]:
import ipaddress
import re
from dataclasses import dataclass
from functools import lru_cache
from urllib.parse import quote, unquote, urlsplit

import tldextract

EXTRACT = tldextract.TLDExtract(
    suffix_list_urls=(),
    cache_dir=None,
    include_psl_private_domains=True,
)
EMAIL = re.compile(r"[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}")
PHONE = re.compile(r"(?<!\w)(?:\+62|62|0)[ .-]?8[1-9](?:[ .-]?\d){6,10}\b")
NUMBER = re.compile(r"(?<!\d)(?:\d[ -]?){8,}(?!\d)")
TOKEN = re.compile(
    r"\d{9,10}:[\w-]{30,}|[a-fA-F0-9]{24,}|"
    r"[A-Za-z0-9_-]{24,}\.[A-Za-z0-9_-]{16,}(?:\.[A-Za-z0-9_-]+)?|"
    r"(?=[A-Za-z0-9_-]{24,}(?:\W|$))(?=[A-Za-z0-9_-]*\d)[A-Za-z0-9_-]{24,}"
)
SENSITIVE_KEY = re.compile(
    r"^(?:password|passwd|pwd|token|secret|otp|pin|session|authorization|"
    r"email|phone|mobile|nik|account|credential|access_token|api_key)$",
    re.I,
)
CONTROL = re.compile(r"[\x00-\x20\x7f\\]")
HOST_LABEL = re.compile(r"^[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?$", re.I)


class InvalidUrl(ValueError):
    """Only a fixed reason code; never include user input in the exception."""


@dataclass(frozen=True)
class PreparedUrl:
    core: str
    host: str
    path: str
    scheme: str
    group: str
    suffix: str
    redactions: int

    @property
    def digest(self) -> str:
        return hashlib.sha256(self.core.encode()).hexdigest()


@lru_cache(maxsize=100_000)
def domain_parts(host: str) -> tuple[str, str]:
    result = EXTRACT(host)
    return result.top_domain_under_public_suffix or host, result.suffix or "unknown"


def redact_path(path: str) -> tuple[str, int]:
    decoded = unquote(unquote(path))
    if CONTROL.search(decoded.replace(" ", "")):
        raise InvalidUrl("decoded_control_or_backslash")
    if re.search(r"%[0-9a-fA-F]{2}", decoded):
        raise InvalidUrl("ambiguous_nested_encoding")
    count = 0
    # A replacement can join surrounding token pieces into a new match.
    for _ in range(5):
        changed = 0
        for pattern in (EMAIL, PHONE, NUMBER, TOKEN):
            decoded, matches = pattern.subn("REDACTED", decoded)
            count += matches
            changed += matches
        if not changed:
            break
    segments = decoded.split("/")
    for index in range(1, len(segments)):
        if SENSITIVE_KEY.fullmatch(segments[index - 1]) and segments[index]:
            segments[index] = "REDACTED"
            count += 1
    # Fixed escaping avoids storing decoded data as a new query/fragment.
    encoded = quote("/".join(segments), safe="/-._~!$&'()*+,;=:@")
    if any(pattern.search(encoded) for pattern in (EMAIL, PHONE, NUMBER, TOKEN)):
        raise InvalidUrl("residual_sensitive_candidate")
    return encoded, count


def prepare_url(value: str) -> PreparedUrl:
    if not isinstance(value, str) or not value.strip():
        raise InvalidUrl("empty")
    value = value.strip()
    if len(value) > 16_384:
        raise InvalidUrl("too_long")
    if CONTROL.search(value):
        raise InvalidUrl("control_whitespace_or_backslash")
    if "://" not in value:
        if re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*:", value) and not re.match(
            r"^[^/:]+:\d+(?:/|$)",
            value,
        ):
            raise InvalidUrl("unsupported_scheme")
        value = "https://" + value
    try:
        parsed = urlsplit(value)
        scheme = parsed.scheme.lower()
        if scheme not in ("http", "https"):
            raise InvalidUrl("unsupported_scheme")
        if parsed.username is not None or parsed.password is not None:
            raise InvalidUrl("userinfo")
        host = (
            (parsed.hostname or "").rstrip(".").encode("idna").decode("ascii").lower()
        )
        port = parsed.port
    except (ValueError, UnicodeError):
        raise InvalidUrl("invalid_authority") from None
    if not host or len(host) > 253:
        raise InvalidUrl("invalid_host")
    try:
        address = ipaddress.ip_address(host)
    except ValueError:
        address = None
    if address is not None:
        if not address.is_global:
            raise InvalidUrl("non_public_ip")
    elif "." not in host or not all(
        HOST_LABEL.fullmatch(part) for part in host.split(".")
    ):
        raise InvalidUrl("invalid_or_local_host")
    elif host.endswith((".localhost", ".local", ".internal", ".test", ".invalid")):
        raise InvalidUrl("non_public_host")
    if PHONE.search(host) or NUMBER.search(host) or TOKEN.search(host):
        raise InvalidUrl("sensitive_host_candidate")
    group, suffix = domain_parts(host)
    if address is not None:
        group, suffix = host, "ip"
    authority = f"[{host}]" if ":" in host else host
    if port is not None and (scheme, port) not in (("http", 80), ("https", 443)):
        authority += f":{port}"
    path, count = redact_path(parsed.path or "/")
    return PreparedUrl(
        f"{scheme}://{authority}{path}", host, path, scheme, group, suffix, count
    )

In [3]:
def wilson(successes, total):
    if total == 0:
        return [None, None]
    z = 1.959963984540054
    rate = successes / total
    denominator = 1 + z * z / total
    center = (rate + z * z / (2 * total)) / denominator
    half = (
        z
        * np.sqrt(rate * (1 - rate) / total + z * z / (4 * total * total))
        / denominator
    )
    return [
        0.0 if successes == 0 else float(max(0, center - half)),
        1.0 if successes == total else float(min(1, center + half)),
    ]


def operating_point(y, probability, budget=0.001):
    y = np.asarray(y)
    probability = np.asarray(probability, dtype=float)
    fpr, recall, thresholds = roc_curve(y, probability, drop_intermediate=False)
    eligible = np.flatnonzero(fpr <= budget)
    # Thresholds are descending; the last eligible point maximizes sensitivity.
    index = eligible[-1]
    threshold = float(thresholds[index])
    if not np.isfinite(threshold):
        threshold = float(np.nextafter(1.0, 2.0))
    return {
        "threshold": threshold,
        "empirical_fpr": float(fpr[index]),
        "recall": float(recall[index]),
        "target_fpr": budget,
    }


def metrics(y, probability, threshold):
    y = np.asarray(y, dtype=int)
    probability = np.asarray(probability, dtype=float)
    assert np.isfinite(probability).all()
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()

    def ratio(a, b):
        return float(a / b) if b else 0.0

    result = {
        "rows": len(y),
        "benign": int(tn + fp),
        "phishing": int(tp + fn),
        "threshold": float(threshold),
        "confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
        },
        "accuracy": ratio(tp + tn, len(y)),
        "precision": ratio(tp, tp + fp),
        "recall": ratio(tp, tp + fn),
        "fpr": ratio(fp, tn + fp),
        "f1": ratio(2 * tp, 2 * tp + fp + fn),
        "brier_score": float(brier_score_loss(y, probability)),
        "log_loss": float(
            log_loss(y, np.clip(probability, 1e-7, 1 - 1e-7), labels=[0, 1])
        ),
        "fpr_wilson95_rows_iid_assumption": wilson(fp, fp + tn),
        "recall_wilson95_rows_iid_assumption": wilson(tp, tp + fn),
    }
    if len(np.unique(y)) == 2:
        result["average_precision"] = float(average_precision_score(y, probability))
        result["roc_auc"] = float(roc_auc_score(y, probability))
    result["balanced_accuracy"] = (result["recall"] + 1 - result["fpr"]) / 2
    bins = []
    bucket = np.minimum((probability * 10).astype(int), 9)
    for index in range(10):
        lower = index / 10
        subset = bucket == index
        if subset.any():
            bins.append(
                {
                    "lower": round(float(lower), 1),
                    "rows": int(subset.sum()),
                    "mean_probability": float(probability[subset].mean()),
                    "phishing_fraction": float(y[subset].mean()),
                }
            )
    result["reliability_bins"] = bins
    return result


def group_bootstrap(y, probability, groups, threshold, repeats=200):
    import pandas as pd

    y = np.asarray(y)
    positive = probability >= threshold
    frame = pd.DataFrame(
        {
            "group": groups,
            "tn": (y == 0) & ~positive,
            "fp": (y == 0) & positive,
            "fn": (y == 1) & ~positive,
            "tp": (y == 1) & positive,
        }
    )
    counts = (
        frame.groupby("group", sort=True)[["tn", "fp", "fn", "tp"]].sum().to_numpy()
    )
    rng = np.random.default_rng(20261101)
    draws = []
    for _ in range(repeats):
        tn, fp, fn, tp = counts[rng.integers(0, len(counts), len(counts))].sum(axis=0)
        draws.append([fp / max(1, fp + tn), tp / max(1, tp + fn), tp / max(1, tp + fp)])
    intervals = np.quantile(draws, [0.025, 0.975], axis=0)
    return {
        "resampling_unit": "PSL registered domain/private tenant",
        "repeats": repeats,
        "fpr95": intervals[:, 0].tolist(),
        "recall95": intervals[:, 1].tolist(),
        "precision95": intervals[:, 2].tolist(),
    }


BRANDS = (
    "bca",
    "bni",
    "bri",
    "mandiri",
    "dana",
    "gopay",
    "ovo",
    "shopee",
    "tokopedia",
    "paypal",
    "microsoft",
    "google",
    "apple",
    "facebook",
)

KEYWORDS = (
    "login",
    "signin",
    "verify",
    "secure",
    "account",
    "update",
    "password",
    "confirm",
    "bank",
    "wallet",
    "hadiah",
    "klaim",
    "undangan",
)

NUMERIC = (
    "url_length",
    "host_length",
    "path_length",
    "host_labels",
    "host_hyphens",
    "host_digits",
    "host_digit_ratio",
    "host_entropy",
    "path_entropy",
    "path_depth",
    "path_digits",
    "path_digit_ratio",
    "path_hyphens",
    "path_underscores",
    "path_dots",
    "path_percents",
    "host_max_label",
    "registered_length",
    "subdomain_length",
    "suffix_length",
    "punycode",
    "ip_literal",
    "https",
    "keyword_count",
    "brand_count",
    "brand_in_subdomain",
    "brand_in_path",
    "path_equals",
    "path_at",
    "path_semicolon",
    "path_upper_ratio",
    "path_non_ascii_encoded",
    "download_extension",
    "host_vowel_ratio",
)


def entropy(text):
    if not text:
        return 0.0
    return -sum(
        (count / len(text)) * math.log2(count / len(text))
        for count in Counter(text).values()
    )


def feature_row(parsed: PreparedUrl):
    host, path = parsed.host, parsed.path
    low = parsed.core.lower()
    sub = host[: -len(parsed.group)].rstrip(".") if host.endswith(parsed.group) else ""
    path_low = path.lower()
    digits_host = sum(char.isdigit() for char in host)
    digits_path = sum(char.isdigit() for char in path)
    values = [
        len(parsed.core),
        len(host),
        len(path),
        host.count(".") + 1,
        host.count("-"),
        digits_host,
        digits_host / len(host),
        entropy(host),
        entropy(path),
        len([part for part in path.split("/") if part]),
        digits_path,
        digits_path / max(len(path), 1),
        path.count("-"),
        path.count("_"),
        path.count("."),
        path.count("%"),
        max(map(len, host.split("."))),
        len(parsed.group),
        len(sub),
        len(parsed.suffix),
        int("xn--" in host),
        int(parsed.suffix == "ip"),
        int(parsed.scheme == "https"),
        sum(word in low for word in KEYWORDS),
        sum(word in low for word in BRANDS),
        sum(word in sub for word in BRANDS),
        sum(word in path_low for word in BRANDS),
        path.count("="),
        path.count("@"),
        path.count(";"),
        sum(char.isupper() for char in path) / max(len(path), 1),
        path_low.count("%c") + path_low.count("%e"),
        int(path_low.endswith((".apk", ".exe", ".zip", ".scr"))),
        sum(char in "aeiou" for char in host) / len(host),
    ]
    return dict(zip(NUMERIC, values)) | {"suffix": parsed.suffix}


NUMERIC_V2 = tuple(name for name in NUMERIC if name != "https")


def model_text(cores):
    return (
        pd.Series(cores)
        .reset_index(drop=True)
        .str.replace(r"^https?://", "", regex=True)
    )


def numeric_values(cores):
    def numeric_row(core):
        prepared = prepare_url(core)
        assert prepared.core == core, "Core sanitization audit failed."
        result = feature_row(prepared)
        result["url_length"] = len(prepared.core.split("://", 1)[1])
        return result

    records = (numeric_row(core) for core in cores)
    data = pd.DataFrame.from_records(records)
    values = data[list(NUMERIC_V2)].to_numpy(dtype=np.float32)
    return np.log1p(np.clip(values, 0, None))


def feature_matrix(encoder, cores, numeric=None):
    text = model_text(cores)
    parts = [encoder["char"].transform(text)]
    if encoder["word"] is not None:
        parts.append(CONFIG["word_weight"] * encoder["word"].transform(text))
        values = numeric_values(cores) if numeric is None else numeric
        values = np.clip(encoder["scaler"].transform(values), 0, 10)
        parts.append(CONFIG["numeric_weight"] * csr_matrix(normalize(values)))
    return hstack(parts, format="csr", dtype=np.float32)


def new_raw(bundle, cores):
    cores = pd.Series(cores).reset_index(drop=True)
    values = []
    for start in range(0, len(cores), 4096):
        values.append(
            bundle["classifier"].predict_proba(
                feature_matrix(bundle["encoder"], cores.iloc[start : start + 4096])
            )[:, 1]
        )
    return np.concatenate(values)


def new_probability(bundle, cores):
    raw = np.clip(new_raw(bundle, cores), 1e-7, 1 - 1e-7)
    return bundle["calibrator"].predict_proba(logit(raw).reshape(-1, 1))[:, 1]


def source_capped_point(labels, probability, sources, budget):
    labels, probability, sources = (
        np.asarray(labels),
        np.asarray(probability),
        np.asarray(sources),
    )
    points = {
        source: operating_point(
            labels[sources == source], probability[sources == source], budget
        )
        for source in sorted(set(sources))
    }
    threshold = max(point["threshold"] for point in points.values())
    rates = {
        source: metrics(
            labels[sources == source], probability[sources == source], threshold
        )
        for source in points
    }
    return {
        "threshold": threshold,
        "target_fpr_per_source": budget,
        "source_rates": rates,
    }


def tune_diagnostics(labels, probability, groups, sources):
    point = source_capped_point(labels, probability, sources, 0.01)
    high = source_capped_point(labels, probability, sources, 0.001)
    recalls = [rate["recall"] for rate in point["source_rates"].values()]
    overall = metrics(labels, probability, point["threshold"])
    return {
        "macro_source_recall": float(np.mean(recalls)),
        "min_source_recall": min(recalls),
        "source_recalls": {s: r["recall"] for s, r in point["source_rates"].items()},
        "recall_at_1pct": overall["recall"],
        "recall_at_0_1pct": metrics(labels, probability, high["threshold"])["recall"],
        "average_precision": float(average_precision_score(labels, probability)),
        "selection_score": [
            float(np.mean(recalls)),
            min(recalls),
            float(average_precision_score(labels, probability)),
        ],
    }


_scheme_probe = ["http://example.org/login", "https://example.org/login"]


def paired_domain_interval(data, p_old, p_new, t_old, t_new, repeats=200):
    y = data.label.to_numpy()
    records = pd.DataFrame({"group": data.group.to_numpy()})
    for prefix, p, t in (("old", p_old, t_old), ("new", p_new, t_new)):
        pred = p >= t
        records[prefix + "_tp"] = (y == 1) & pred
        records[prefix + "_tn"] = (y == 0) & ~pred
        records[prefix + "_fp"] = (y == 0) & pred
        records[prefix + "_fn"] = (y == 1) & ~pred
    values = records.groupby("group", sort=True).sum().to_numpy()
    rng, draws = np.random.default_rng(SEED), []
    for _ in range(repeats):
        old_tp, old_tn, old_fp, old_fn, new_tp, new_tn, new_fp, new_fn = values[
            rng.integers(0, len(values), len(values))
        ].sum(axis=0)
        draws.append(
            [
                (new_tp + new_tn) / max(1, new_tp + new_tn + new_fp + new_fn)
                - (old_tp + old_tn) / max(1, old_tp + old_tn + old_fp + old_fn),
                new_tp / max(1, new_tp + new_fn) - old_tp / max(1, old_tp + old_fn),
                new_fp / max(1, new_fp + new_tn) - old_fp / max(1, old_fp + old_tn),
            ]
        )
    interval = np.quantile(draws, [0.025, 0.975], axis=0)
    return {
        "accuracy_delta95": interval[:, 0].tolist(),
        "recall_delta95": interval[:, 1].tolist(),
        "fpr_delta95": interval[:, 2].tolist(),
        "resampling_unit": "domain/private tenant",
        "repeats": repeats,
    }


def fit_encoder(view, cores, sources):
    text = model_text(cores)
    source_series = pd.Series(sources).reset_index(drop=True)
    pieces = [
        text[source_series.eq(source)].sample(
            min(CONFIG["vocabulary_source_cap"], int(source_series.eq(source).sum())),
            random_state=SEED,
        )
        for source in sorted(source_series.unique())
    ]
    sample = pd.concat(pieces)
    if len(sample) > CONFIG["vocabulary_fit_limit"]:
        sample = sample.sample(CONFIG["vocabulary_fit_limit"], random_state=SEED)
    rich = "word" in view
    encoder = {
        "view": view,
        "char": TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 5) if rich else (3, 5),
            max_features=100000 if rich else 40000,
            min_df=CONFIG["char_min_df"],
            sublinear_tf=True,
            dtype=np.float32,
        ),
        "word": None,
        "scaler": None,
    }
    encoder["char"].fit(sample)
    numeric = None
    if rich:
        encoder["word"] = TfidfVectorizer(
            analyzer="word",
            token_pattern=r"(?u)\b[a-zA-Z]{2,}\b",
            ngram_range=(1, 2),
            max_features=CONFIG["word_max_features"],
            min_df=4,
            sublinear_tf=True,
            dtype=np.float32,
        ).fit(sample)
        numeric = numeric_values(cores)
        encoder["scaler"] = StandardScaler(with_mean=False).fit(numeric)
    return encoder, feature_matrix(encoder, cores, numeric)

In [4]:
def positive_sensitivity(data, probability, threshold):
    predicted = probability >= threshold
    positives = int(predicted.sum())
    return {
        "rows": len(data),
        "groups": int(data.group.nunique()),
        "tp": positives,
        "fn": len(data) - positives,
        "threshold": float(threshold),
        "recall": positives / len(data),
        "recall_wilson95_rows_iid_assumption": wilson(positives, len(data)),
        "fpr": None,
        "traffic_precision": None,
        "binary_accuracy": None,
        "limitation": "positive_only_no_benign_denominator",
    }


def positive_paired_domain_interval(
    data, old_p, new_p, old_threshold, new_threshold, repeats=200
):
    counts = (
        pd.DataFrame(
            {
                "group": data.group.to_numpy(),
                "n": 1,
                "old_tp": old_p >= old_threshold,
                "new_tp": new_p >= new_threshold,
            }
        )
        .groupby("group", sort=True)[["n", "old_tp", "new_tp"]]
        .sum()
        .to_numpy()
    )
    rng, draws = np.random.default_rng(SEED), []
    for _ in range(repeats):
        n, old_tp, new_tp = counts[rng.integers(0, len(counts), len(counts))].sum(
            axis=0
        )
        draws.append((new_tp - old_tp) / n)
    return {
        "recall_delta95": np.quantile(draws, [0.025, 0.975]).tolist(),
        "resampling_unit": "registered_domain_private_tenant",
        "repeats": repeats,
        "fpr_interval": None,
    }

In [5]:
protocol = json.loads((REPORTS / "low-fpr-control-protocol.json").read_text())
baseline_frozen = json.loads((V3 / "reports/frozen-v3.json").read_text())
assert (
    sha256_file(V3 / "artifacts/url-core-logistic-v3.joblib")
    == baseline_frozen["bundle_sha256"]
)
assert {name: version(name) for name in baseline_frozen["versions"]} == baseline_frozen[
    "versions"
]
baseline = joblib.load(V3 / "artifacts/url-core-logistic-v3.joblib")
CONFIG = baseline["metadata"]["config"]
frozen = json.loads((REPORTS / "frozen-v6.json").read_text())
assert (
    sha256_file(RUN / "artifacts/url-core-logistic-v6.joblib")
    == frozen["bundle_sha256"]
)
bundle = joblib.load(RUN / "artifacts/url-core-logistic-v6.joblib")
threshold_data = pd.read_parquet(DATA / "development-v6.parquet")
threshold_data = threshold_data[threshold_data.partition.eq("threshold")]
with threadpool_limits(limits=4):
    probability = new_probability(baseline, threshold_data.core)
control_points = {
    "warning": source_capped_point(
        threshold_data.label, probability, threshold_data.training_source, 0.01
    ),
    "high": source_capped_point(
        threshold_data.label, probability, threshold_data.training_source, 0.001
    ),
}
control_thresholds = {key: value["threshold"] for key, value in control_points.items()}
control_frozen = {
    "frozen_at_utc": datetime.now(timezone.utc).isoformat(),
    "protocol": protocol,
    "baseline_bundle_sha256": baseline_frozen["bundle_sha256"],
    "model_v6_bundle_sha256": frozen["bundle_sha256"],
    "development_sha256": frozen["development_sha256"],
    "thresholds": control_thresholds,
    "threshold_partition_diagnostics": control_points,
    "late_declared_analysis": True,
    "new_test_not_used_to_select_control_threshold": True,
}
write_json(REPORTS / "low-fpr-control-frozen.json", control_frozen)
print("CONTROL_THRESHOLDS_FROZEN", control_thresholds, flush=True)
policies = {
    "v3_original": baseline["metadata"]["thresholds"],
    "v3_development_threshold_control": control_thresholds,
    "v6": bundle["metadata"]["thresholds"],
}
binary = pd.read_parquet(DATA / "deepurlbench-holdout-v6.parquet")
positive = pd.read_parquet(DATA / "phishtank-2026-v6.parquet")
with threadpool_limits(limits=4):
    old_binary, new_binary = (
        new_probability(baseline, binary.core),
        new_probability(bundle, binary.core),
    )
    old_positive, new_positive = (
        new_probability(baseline, positive.core),
        new_probability(bundle, positive.core),
    )
binary_results, positive_results = {}, {}
for name, policy in policies.items():
    binary_p = new_binary if name == "v6" else old_binary
    positive_p = new_positive if name == "v6" else old_positive
    binary_results[name] = {
        level: metrics(binary.label, binary_p, t) for level, t in policy.items()
    }
    positive_results[name] = {
        level: positive_sensitivity(positive, positive_p, t)
        for level, t in policy.items()
    }
binary_interval = paired_domain_interval(
    binary,
    old_binary,
    new_binary,
    control_thresholds["warning"],
    policies["v6"]["warning"],
)
positive_interval = positive_paired_domain_interval(
    positive,
    old_positive,
    new_positive,
    control_thresholds["warning"],
    policies["v6"]["warning"],
)
report = {
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "late_declared_analysis": True,
    "protocol": protocol,
    "control_frozen": control_frozen,
    "control_notebook": NOTEBOOK_NAME,
    "control_code_sha256": control_code_sha256,
    "binary_new_domain_holdout": binary_results,
    "community_2026_positive_sensitivity": positive_results,
    "paired_binary_control_vs_v6_domain_intervals": binary_interval,
    "paired_community_control_vs_v6_recall_interval": positive_interval,
    "v6_model_or_threshold_changed": False,
    "limitations": [
        "Analysis added after main evaluation started; not initial preregistration.",
        "Control threshold selected only from development threshold partition, never from test.",
        "Original v3 policy remains reported; low-FPR control is an additional reference.",
        "Vendor-control FPR and community-phishing sensitivity are different distributions.",
        "No new model/policy selected from these results and no production promotion.",
    ],
}
write_json(ROOT / "ml/reports/v6-low-fpr-control.json", report)
for name in policies:
    b = binary_results[name]["warning"]
    p = positive_results[name]["warning"]
    print(
        name,
        "vendor recall",
        b["recall"],
        "vendor fpr",
        b["fpr"],
        "community recall",
        p["recall"],
        flush=True,
    )

display(
    pd.DataFrame(
        [
            {
                "reference": name,
                "vendor_recall_pct": 100 * binary_results[name]["warning"]["recall"],
                "vendor_fpr_pct": 100 * binary_results[name]["warning"]["fpr"],
                "community_2026_recall_pct": 100
                * positive_results[name]["warning"]["recall"],
                "community_2026_fpr": "N/A",
            }
            for name in policies
        ]
    ).round(4)
)
print("Late-declared supplementary analysis; no model/threshold promotion.")

CONTROL_THRESHOLDS_FROZEN {'warning': 0.9928418787439804, 'high': 0.999096558753516}


v3_original vendor recall 0.535052012663953 vendor fpr 0.05227949856652988 community recall 0.5547524004747006


v3_development_threshold_control vendor recall 0.1926729986431479 vendor fpr 0.010062398111192309 community recall 0.26594023087711727


v6 vendor recall 0.4631388511985527 vendor fpr 0.009444038450727978 community recall 0.4653144891574064


,reference,vendor_recall_pct,vendor_fpr_pct,community_2026_recall_pct,community_2026_fpr
0,v3_original,53.5052,5.2279,55.4752,N/A
1,v3_development_threshold_control,19.2673,1.0062,26.5940,N/A
2,v6,46.3139,0.9444,46.5314,N/A


Late-declared supplementary analysis; no model/threshold promotion.


Kontrol vendor dan phishing komunitas berasal dari populasi berbeda. Interval FPR masih dapat melintasi 1% ketika point estimate di bawah 1%. Kebijakan asli juga wajib dilaporkan: menekan false alarm dapat menurunkan recall mentah. Audit manusia kecil serta regresi sumber tetap terlihat di laporan utama; kontrol tambahan tidak menutupi keterbatasan itu.